# Detector de prácticas duplicadas · prototipo

**Flujo:** 1 Extractor → 2 Pre-procesado → 3 Clasificación (reglas · 3b LLM opcional) → 4 Cruzador (ficticio) → 5 Trazador

- **Hallazgo** = un grupo de filas. Todos son *potenciales*: sin paciente no se pueden confirmar.
- **Métricas:** cada bloque registra latencia, páginas, tokens, costo y aciertos de caché en `METRICAS`.
- **Idempotencia:** caché por hash (PDF + páginas / prompt), ids derivados del contenido e `INSERT OR IGNORE`. Correrlo dos veces da los mismos hallazgos, y la segunda corrida cuesta USD 0.

Subí los 3 PDFs a `/content` y corré todo.

**Antes de correr:** cargá `DOCAI_ENDPOINT` y `GCP_PROJECT` en los Secretos de Colab (🔑), o como variables de entorno.

In [ ]:
# --- Configuración ------------------------------------------------------------------------------
%pip install -q pymupdf google-genai

import base64, hashlib, json, os, math, random, re, sqlite3, time, unicodedata
from bisect import bisect_right
from collections import defaultdict
from contextlib import contextmanager
from decimal import Decimal, InvalidOperation
from pathlib import Path
from statistics import median

import pandas as pd
import pymupdf
from IPython.display import Image, display

PDFS = [
    "/content/detalle_(1)_ANONIMIZADO_sin_afiliado.pdf",
    "/content/detalle_(2)_ANONIMIZADO_sin_afiliado.pdf",
    "/content/detalle_ANONIMIZADO_sin_afiliado.pdf",
]
# Datos de GCP: Secretos de Colab (🔑) o variables de entorno. No van escritos en el código.
def secreto(nombre, defecto=""):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre, defecto)


ENDPOINT = secreto("DOCAI_ENDPOINT")   # https://us-documentai.googleapis.com/v1/projects/<PROYECTO>/locations/us/processors/<PROCESADOR>:process
FIELD_MASK = "text,pages.pageNumber,pages.dimension,pages.tokens"
PAGES_PER_REQUEST = 15    # tope del modo online del OCR
MAX_RETRIES = 3

MODEL = "gemini-3.8-flash"
PROJECT_ID = secreto("GCP_PROJECT")
LOCATION = "global"
USAR_LLM = False          # True: el flujo 3b (LLM + skill) reemplaza a la regla de duplicados

# Precios en USD. Verificalos antes de usar los números.
PRICE_OCR = 1.50 / 1000             # Enterprise Document OCR, por página
PRICE_IN, PRICE_OUT = 0.75, 3.75    # gemini-3.8-flash, por 1M tokens

UMBRAL_P = 0.05           # duplicado potencial si P(≥2 el mismo día) < 5 %
UMBRAL_P_LLM = 0.30       # filtro laxo: candidatos que se le mandan al LLM
REGLAS_VERSION = f"reglas-v1 p<{UMBRAL_P}"

OUT = Path("/content/salida")
CACHE = OUT / "cache"
DB = OUT / "detector.sqlite"
CACHE.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 90)

In [ ]:
# --- Métricas y utilidades -----------------------------------------------------------------------
METRICAS = {}   # (bloque, detalle) -> fila; re-ejecutar una celda pisa su fila


@contextmanager
def medir(bloque, detalle=""):
    m = {"bloque": bloque, "detalle": detalle, "latencia_s": 0.0, "paginas": 0,
         "tokens_in": 0, "tokens_out": 0, "costo_usd": 0.0, "cache_hits": 0}
    t0 = time.perf_counter()
    try:
        yield m
    finally:
        m["latencia_s"] = round(time.perf_counter() - t0, 3)
        METRICAS[(bloque, detalle)] = m


def sha(*parts, n=16):
    """Hash corto y estable: base de la caché y de los ids."""
    return hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:n]

## 1. Extractor
Parser de la planilla: el mismo del notebook anterior. Solo se agregó `x0`/`x1` a cada renglón para que el trazador tenga la caja completa.

In [ ]:
# --- Parser de planilla (Document AI -> una fila por renglón) -----------------------------------
ROW_TOL = 0.6       # misma fila si los centros difieren menos que ROW_TOL × alto de token

# Títulos de izquierda a derecha.
# "left": el contenido arranca donde arranca el título (texto).
# "right": el contenido termina donde termina el título (números).
HEADERS = [
    ("nro_int",      "Nro. Int.",           "left"),
    ("nro_afiliado", "Nro. de Afiliado",    "left"),
    ("nro_bono",     "Nro. de Bono",        "left"),
    ("profesional",  "Profesional Efector", "left"),
    ("servicio",     "Servicio",            "left"),
    ("fecha",        "Fecha",               "left"),
    ("paciente",     "Apellido y Nombre",   "left"),
    ("prestacion",   "Prestación",          "left"),
    ("cant",         "Cant.",               "right"),
    ("der",          "Der.",                "right"),
    ("hon",          "Hon.",                "right"),
    ("total",        "Total",               "right"),
]
FIELDS = [n for n, _, _ in HEADERS]

# El OCR a veces devuelve letras griegas iguales a las latinas ("ΜΑPA").
GREEK = str.maketrans("ΑΒΕΖΗΙΚΜΝΟΡΤΥΧοα", "ABEZHIKMNOPTYXoa")


def key(s):
    """Versión comparable: sin acentos, minúsculas, solo letras y números."""
    s = unicodedata.normalize("NFKD", s.translate(GREEK))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r"[^a-z0-9]", "", s)


def anchor_text(full, layout):
    """Recorta del texto completo el tramo al que apunta el elemento."""
    segs = (layout.get("textAnchor") or {}).get("textSegments") or []
    return "".join(full[int(s.get("startIndex", 0)):int(s.get("endIndex", 0))] for s in segs)


def page_tokens(full, page):
    """Tokens de la página con su texto y su caja normalizada 0..1."""
    out = []
    for t in page.get("tokens", []):
        lay = t.get("layout", {})
        if lay.get("orientation", 0) not in (0, 1, "PAGE_UP", "ORIENTATION_UNSPECIFIED"):
            continue  # texto rotado (logos, sellos), no es la tabla
        v = (lay.get("boundingPoly") or {}).get("normalizedVertices") or []
        if not v:
            continue
        xs = [p.get("x", 0.0) for p in v]
        ys = [p.get("y", 0.0) for p in v]
        raw = anchor_text(full, lay).replace("\n", " ").translate(GREEK)
        if t.get("detectedBreak") and not raw.endswith(" "):
            raw += " "
        out.append({
            "raw": raw, "key": key(raw), "conf": lay.get("confidence", 0.0),
            "x0": min(xs), "x1": max(xs), "y0": min(ys), "y1": max(ys),
            "xc": (min(xs) + max(xs)) / 2, "yc": (min(ys) + max(ys)) / 2,
        })
    return out


def group_rows(toks):
    """Agrupa tokens en filas visuales según su altura en la página."""
    if not toks:
        return []
    h = median(t["y1"] - t["y0"] for t in toks)
    rows = []
    for t in sorted(toks, key=lambda t: t["yc"]):
        if rows and abs(t["yc"] - rows[-1]["yc"]) <= ROW_TOL * h:
            rows[-1]["toks"].append(t)
            rows[-1]["yc"] = median(x["yc"] for x in rows[-1]["toks"])
        else:
            rows.append({"yc": t["yc"], "toks": [t]})
    for r in rows:
        r["toks"].sort(key=lambda t: t["x0"])
    return rows


def match_headers(toks):
    """Busca los títulos, en orden, dentro de una fila. Devuelve {columna: caja del título}."""
    found, i = {}, 0
    for name, label, align in HEADERS:
        target = key(label)
        for j in range(i, len(toks)):
            if not toks[j]["key"]:
                continue
            acc, k = "", j
            while k < len(toks) and len(acc) < len(target) and target.startswith(acc + toks[k]["key"]):
                acc += toks[k]["key"]
                k += 1
            if acc == target:
                span = [t for t in toks[j:k] if t["key"]]
                found[name] = {"align": align,
                               "x0": min(t["x0"] for t in span), "x1": max(t["x1"] for t in span)}
                i = k
                break
    return found


def column_cuts(found):
    """Límites horizontales entre columnas, según cómo se alinea cada una."""
    cols = sorted(found.items(), key=lambda kv: kv[1]["x0"])
    cuts = []
    for (_, a), (_, b) in zip(cols, cols[1:]):
        if a["align"] == "right" and b["align"] == "right":
            cuts.append(a["x1"])                    # entre números: donde termina el título izquierdo
        elif a["align"] == "right":
            cuts.append((a["x1"] + b["x0"]) / 2)    # número y después texto: a mitad de camino
        else:
            cuts.append(b["x0"])                    # después de texto: donde arranca el título siguiente
    return [n for n, _ in cols], cuts


def clean(s):
    """Saca el ruido de los tachados ('|||', '....') y espacios de más."""
    s = re.sub(r"\s+", " ", s.replace("|", " ")).strip()
    return s.strip(" .")


def parse_page(full, page):
    n_page = page.get("pageNumber")
    rows = group_rows(page_tokens(full, page))
    header = max(rows, key=lambda r: len(match_headers(r["toks"])), default=None)
    found = match_headers(header["toks"]) if header else {}
    if len(found) < len(HEADERS) // 2:
        print(f"Página {n_page}: no encontré la fila de títulos")
        return []
    names, cuts = column_cuts(found)
    top = max(t["y1"] for t in header["toks"])
    out = []
    for r in rows:
        if r["yc"] <= top:
            continue  # encabezado de la página y fila de títulos
        if re.search(r"pagina\d+de\d+", "".join(t["key"] for t in r["toks"])):
            continue  # pie "Página X de N"
        cells = dict.fromkeys(names, "")
        for t in r["toks"]:
            cells[names[bisect_right(cuts, t["xc"])]] += t["raw"]
        cells = {n: clean(v) for n, v in cells.items()}
        if not any(cells.values()):
            continue
        cells.update(
            pagina=n_page,
            x0=round(min(t["x0"] for t in r["toks"]), 4),
            x1=round(max(t["x1"] for t in r["toks"]), 4),
            y0=round(min(t["y0"] for t in r["toks"]), 4),
            y1=round(max(t["y1"] for t in r["toks"]), 4),
            conf_min=round(min(t["conf"] for t in r["toks"]), 3),
        )
        out.append(cells)
    return out


def ar_decimal(s):
    """'144283,00' o '1.234,56' -> Decimal; vacío o ilegible -> None."""
    s = (s or "").replace("$", "").replace(" ", "")
    try:
        return Decimal(s.replace(".", "").replace(",", ".")) if s else None
    except InvalidOperation:
        return None

### 1a. Splitter
¿La página trae capa de texto o es solo imagen? Las de texto irían a un parser interno sin costo de OCR; las de imagen, a Document AI.

In [ ]:
# --- 1a. Splitter: texto o imagen, página por página ---------------------------------------------
MIN_CHARS = 20   # menos caracteres que esto = página escaneada

LOTE = {}
with medir("1a splitter") as m:
    for path in PDFS:
        with pymupdf.open(path) as doc:
            rutas = {i: "texto" if len(p.get_text().strip()) >= MIN_CHARS else "imagen"
                     for i, p in enumerate(doc)}
        LOTE[path] = {"path": path, "archivo": Path(path).name, "rutas": rutas,
                      "sha": hashlib.sha256(Path(path).read_bytes()).hexdigest()}
        m["paginas"] += len(rutas)

display(pd.DataFrame([{
    "archivo": x["archivo"], "sha256": x["sha"][:12], "paginas": len(x["rutas"]),
    "texto": sum(r == "texto" for r in x["rutas"].values()),
    "imagen": sum(r == "imagen" for r in x["rutas"].values()),
} for x in LOTE.values()]).set_index("archivo"))

### 1b. Procesadores
- **Texto:** parser interno, pendiente (en este lote no hay páginas con texto).
- **Imagen:** Document AI en tramos de 15 páginas. Cada tramo se guarda en caché con la clave `hash(PDF, páginas, endpoint)`. Si el tramo ya está en caché, no se llama a la API y no se cobra.

In [ ]:
# --- 1b. Procesadores ----------------------------------------------------------------------------
import google.auth
from google.auth.transport.requests import AuthorizedSession

_session = None


def docai_session():
    """Sesión con la SA del entorno; se crea solo si hay que llamar a la API."""
    global _session
    if _session is None:
        creds = globals().get("credentials") or google.auth.default()[0]
        if getattr(creds, "requires_scopes", False):
            creds = creds.with_scopes(["https://www.googleapis.com/auth/cloud-platform"])
        _session = AuthorizedSession(creds)
    return _session


def pdf_pages(path, pages):
    """PDF nuevo con solo esas páginas (0-based)."""
    with pymupdf.open(path) as src, pymupdf.open() as out:
        for i in pages:
            out.insert_pdf(src, from_page=i, to_page=i)
        return out.tobytes()


def docai(pdf_bytes):
    """Un request online; reintenta 429/5xx con espera creciente."""
    body = {"rawDocument": {"content": base64.b64encode(pdf_bytes).decode(),
                            "mimeType": "application/pdf"},
            "fieldMask": FIELD_MASK}
    for attempt in range(MAX_RETRIES + 1):
        resp = docai_session().post(ENDPOINT, json=body, timeout=300)
        if resp.ok:
            return resp.json()["document"]
        if resp.status_code not in (429, 500, 502, 503, 504) or attempt == MAX_RETRIES:
            raise RuntimeError(f"Document AI {resp.status_code}: {resp.text[:300]}")
        time.sleep(2 ** attempt + random.random())


def procesar_texto(lote):
    """Páginas con capa de texto -> parser interno (pendiente)."""
    pages = [i for i, r in lote["rutas"].items() if r == "texto"]
    if pages:
        print(f"{lote['archivo']}: {len(pages)} páginas con texto sin procesar (parser interno pendiente)")
    return []


def procesar_imagenes(lote, m):
    """Páginas imagen -> Document AI con caché. Devuelve los renglones parseados."""
    pages = [i for i, r in lote["rutas"].items() if r == "imagen"]
    filas = []
    for k in range(0, len(pages), PAGES_PER_REQUEST):
        chunk = pages[k:k + PAGES_PER_REQUEST]
        f = CACHE / f"docai_{sha(lote['sha'], chunk, ENDPOINT, FIELD_MASK)}.json"
        if f.exists():
            doc = json.loads(f.read_text(encoding="utf-8"))
            m["cache_hits"] += 1
        else:
            doc = docai(pdf_pages(lote["path"], chunk))
            f.write_text(json.dumps(doc), encoding="utf-8")
            m["costo_usd"] += len(chunk) * PRICE_OCR
        m["paginas"] += len(chunk)
        for p in doc.get("pages", []):
            p["pageNumber"] = chunk[p["pageNumber"] - 1] + 1   # número de página en el PDF original
            filas += parse_page(doc["text"], p)
    return filas

In [ ]:
# --- 1c. Extracción del lote ---------------------------------------------------------------------
CRUDO = []
resumen = []
for lote in LOTE.values():
    with medir("1b extracción", lote["archivo"]) as m:
        filas = procesar_texto(lote) + procesar_imagenes(lote, m)
        for f in filas:
            f["archivo"], f["sha"] = lote["archivo"], lote["sha"]
        CRUDO += filas
    resumen.append({"archivo": lote["archivo"], "renglones": len(filas), "paginas": m["paginas"],
                    "tramos_en_cache": m["cache_hits"], "costo_usd": round(m["costo_usd"], 4),
                    "latencia_s": m["latencia_s"]})

display(pd.DataFrame(resumen).set_index("archivo"))
pd.DataFrame(CRUDO).head()

## 2. Pre-procesado
### 2a. Limpieza y normalización
- **Reparación:** la columna "Apellido y Nombre" está tapada. Si trae un código, es la prestación que el parser corrió de columna.
- **Fuera de la tabla:** la fila de TOTAL impreso se guarda aparte para controlar la suma. Los renglones de ruido (sin Nro. Int. ni importe) se descartan.
- **Familia:** agrupa el mismo estudio cargado con códigos de nomencladores distintos, y conserva el código original de cada fila. La descripción canónica de cada código es la más frecuente.

In [ ]:
# --- 2a. Limpieza y normalización ----------------------------------------------------------------
FAMILIAS = {   # mismo estudio, códigos distintos. El resto: cada código es su propia familia.
    "180161": "ECOGRAFIA TRANSVAGINAL", "861219": "ECOGRAFIA TRANSVAGINAL",
    "180106": "ECOGRAFIA MAMARIA",      "861220": "ECOGRAFIA MAMARIA",
    "341018": "UROTOMOGRAFIA",          "341177": "UROTOMOGRAFIA",
    "341012": "TAC OTROS ORGANOS",      "3410121": "TAC OTROS ORGANOS",
    "3410123": "TAC OTROS ORGANOS",     "3410126": "TAC OTROS ORGANOS",
}
CODIGO = re.compile(r"^\d{4,7}(?:X1)?\s")

with medir("2 pre-procesado") as m:
    df = pd.DataFrame(CRUDO)
    for c in FIELDS:
        df[c] = df[c].fillna("") if c in df else ""

    corrida = (df["prestacion"] == "") & df["paciente"].str.match(CODIGO)
    df.loc[corrida, "prestacion"] = df.loc[corrida, "paciente"]
    df.loc[corrida, "paciente"] = ""

    sin_id = df["nro_int"] == ""
    es_total = sin_id & (df["total"] != "")
    TOTALES = {a: ar_decimal(g["total"].iloc[-1]) for a, g in df[es_total].groupby("archivo")}
    ruido = int((sin_id & ~es_total).sum())
    df = df[~sin_id].copy()

    df[["codigo", "descripcion"]] = df["prestacion"].str.extract(r"^(\S+)\s*(.*)$").fillna("")
    for c in ["der", "hon", "total"]:
        df[c] = df[c].map(ar_decimal)
    df["cant"] = pd.to_numeric(df["cant"], errors="coerce").astype("Int64")
    df["dia"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d").fillna("")

    df["familia"] = df["codigo"].map(FAMILIAS).fillna(df["codigo"])
    canon = df.groupby("codigo")["descripcion"].agg(lambda s: s.mode().iat[0])
    df["practica"] = df["codigo"].map(FAMILIAS).fillna(df["codigo"].map(canon))

    # Ids estables: mismo PDF y misma extracción -> mismo id. `orden` sirve para ver filas contiguas.
    df = df.sort_values(["archivo", "pagina", "y0"]).reset_index(drop=True)
    df["renglon"] = df.groupby(["archivo", "pagina"]).cumcount() + 1
    df["orden"] = df.groupby("archivo").cumcount()
    df["fila_id"] = [f"{s[:8]}-p{p:02d}-r{r:02d}" for s, p, r in zip(df["sha"], df["pagina"], df["renglon"])]
    m["paginas"] = int(df.groupby("archivo")["pagina"].nunique().sum())

print(f"{len(df)} filas · {int(corrida.sum())} prestaciones corridas a 'paciente' reparadas · "
      f"{ruido} renglones de ruido descartados · TOTAL impreso en {len(TOTALES)} de {len(LOTE)} archivos")

norm = df.groupby("familia").agg(practica=("practica", "first"),
                                 codigos=("codigo", lambda s: sorted(set(s))),
                                 descripciones=("descripcion", lambda s: sorted(set(s))),
                                 filas=("fila_id", "size"))
print("\nNormalizaciones aplicadas (familias con más de un código o una descripción):")
display(norm[(norm["codigos"].str.len() > 1) | (norm["descripciones"].str.len() > 1)])

### 2b. Base SQLite
`filas` guarda la extracción con la caja de cada renglón. `hallazgos` y `hallazgo_filas` guardan qué filas forman cada hallazgo.

In [ ]:
# --- 2b. Base SQLite -----------------------------------------------------------------------------
SCHEMA = """
CREATE TABLE IF NOT EXISTS filas (
    fila_id TEXT PRIMARY KEY, archivo TEXT, sha TEXT, pagina INT, renglon INT, orden INT,
    nro_int TEXT, fecha TEXT, servicio TEXT, codigo TEXT, descripcion TEXT, familia TEXT,
    cant INT, der TEXT, hon TEXT, total TEXT, x0 REAL, y0 REAL, x1 REAL, y1 REAL, conf_min REAL);
CREATE TABLE IF NOT EXISTS hallazgos (
    hallazgo_id TEXT PRIMARY KEY, tipo TEXT, regla_version TEXT, fecha TEXT, familia TEXT,
    codigos TEXT, veces INT, monto_en_riesgo TEXT, contiguas INT, nro_ints TEXT, ubicacion TEXT,
    motivo TEXT);
CREATE TABLE IF NOT EXISTS hallazgo_filas (
    hallazgo_id TEXT, fila_id TEXT, PRIMARY KEY (hallazgo_id, fila_id));
"""
FILA_COLS = ["fila_id", "archivo", "sha", "pagina", "renglon", "orden", "nro_int", "dia", "servicio",
             "codigo", "descripcion", "familia", "cant", "der", "hon", "total",
             "x0", "y0", "x1", "y1", "conf_min"]


def sql_val(v):
    """Decimal -> texto (sin redondeos de float); tipos de numpy/pandas -> Python."""
    if v is None or v is pd.NA or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, Decimal):
        return str(v)
    return v.item() if hasattr(v, "item") else v


con = sqlite3.connect(DB)
con.executescript(SCHEMA)
with con:
    con.executemany(f"INSERT OR REPLACE INTO filas VALUES ({', '.join('?' * len(FILA_COLS))})",
                    [[sql_val(v) for v in row] for row in df[FILA_COLS].itertuples(index=False)])
print(f"{DB}: {con.execute('SELECT COUNT(*) FROM filas').fetchone()[0]} filas")

### 2c. Tabla de frecuencias
Pregunta: ¿qué tan raro es ver la misma práctica 2+ veces el mismo día?
- λ = renglones / días con actividad en el lote. Con Poisson, P(≥2) = 1 − e^(−λ)·(1 + λ).
- Se divide por los días **del lote**, no por los días en que aparece la práctica, para que los propios duplicados no inflen la frecuencia.
- Las prácticas repetibles por definición (medicamentos X1, "CADA", exposición subsiguiente, por región) no son candidatas.

In [ ]:
# --- 2c. Frecuencias -----------------------------------------------------------------------------
REPETIBLES = re.compile(r"\bCADA\b|SUBSIGUIENTE|SUBISGUIENTE|SEGUND|POR REGIO", re.I)

dias_lote = df.loc[df["dia"] != "", "dia"].nunique()
freq = (df[df["familia"] != ""].groupby("familia")
        .agg(practica=("practica", "first"), renglones=("fila_id", "size"),
             dias_con_practica=("dia", "nunique")))
freq["lambda"] = (freq["renglones"] / dias_lote).round(3)
freq["p_2"] = freq["lambda"].map(lambda l: round(1 - math.exp(-l) * (1 + l), 4))
freq["repetible"] = freq.index.str.endswith("X1") | freq["practica"].str.contains(REPETIBLES)
freq["candidata"] = ~freq["repetible"] & (freq["p_2"] < UMBRAL_P)
freq = freq.sort_values("lambda", ascending=False)
freq.to_sql("frecuencias", con, if_exists="replace")   # foto de la tabla usada en esta corrida

print(f"{dias_lote} días con actividad · {len(freq)} familias · {int(freq['candidata'].sum())} candidatas "
      f"(P(≥2) < {UMBRAL_P:.0%}, no repetibles)")
freq

## 3. Clasificación por reglas
| Tipo | Agrupa por | Monto en riesgo |
|---|---|---|
| `NRO_INT_REPETIDO` | Nro. Int. (en todo el lote) | suma − máximo |
| `DUPLICADO_POTENCIAL` | día + familia candidata, 2+ filas | suma − máximo (se asume legítima la fila más cara) |
| `REDONDEO` | arancel (código + Der. + Hon. + Total) | lo facturado de más × veces |

El control de la suma contra el TOTAL impreso no es un hallazgo. Si no cuadra, lo primero que hay que sospechar es la extracción.

In [ ]:
# --- 3. Clasificación por reglas -----------------------------------------------------------------
def contiguas(g):
    """¿Hay dos filas seguidas en el mismo archivo? Es lo más cercano al paciente que tenemos."""
    o = sorted(zip(g["archivo"], g["orden"]))
    return any(a == c and d - b == 1 for (a, b), (c, d) in zip(o, o[1:]))


def importes(serie):
    return [x for x in serie if isinstance(x, Decimal)]


def monto_en_riesgo(totales):
    t = importes(totales)
    return sum(t) - max(t) if t else Decimal(0)


def hallazgo(tipo, version, g, motivo, monto, fecha="", contig=None):
    ids = sorted(g["fila_id"])
    return {
        "hallazgo_id": sha(version, tipo, *ids), "tipo": tipo, "fecha": fecha,
        "familia": " / ".join(sorted(set(g["practica"]))), "codigos": ", ".join(sorted(set(g["codigo"]))),
        "veces": len(g), "monto_en_riesgo": monto, "contiguas": contig,
        "nro_ints": ", ".join(g["nro_int"]),
        "ubicacion": "; ".join(f"{a} p.{', '.join(map(str, sorted(set(x['pagina']))))}"
                               for a, x in g.groupby("archivo")),
        "motivo": motivo, "regla_version": version, "fila_ids": ids,
    }


def regla_nro_int(df):
    return [hallazgo("NRO_INT_REPETIDO", REGLAS_VERSION, g, f"Nro. Int. {nro} aparece {len(g)} veces",
                     monto_en_riesgo(g["total"]))
            for nro, g in df.groupby("nro_int") if len(g) > 1]


def regla_mismo_dia(df, freq):
    out = []
    cand = df[df["familia"].isin(freq.index[freq["candidata"]]) & (df["dia"] != "")]
    for (dia, fam), g in cand.groupby(["dia", "familia"]):
        if len(g) < 2:
            continue
        f, c = freq.loc[fam], contiguas(g)
        motivo = (f"{f['practica']} {len(g)} veces el {dia}; esperado {f['lambda']:.2f}/día "
                  f"(P(≥2) = {f['p_2']:.1%})" + ("; filas contiguas" if c else ""))
        out.append(hallazgo("DUPLICADO_POTENCIAL", REGLAS_VERSION, g, motivo,
                            monto_en_riesgo(g["total"]), dia, c))
    return out


def regla_redondeo(df):
    trio = [all(isinstance(v, Decimal) for v in x) for x in zip(df["der"], df["hon"], df["total"])]
    d = df[trio]
    d = d[[a + b != t for a, b, t in zip(d["der"], d["hon"], d["total"])]]
    out = []
    for (cod, der, hon, tot), g in d.groupby(["codigo", "der", "hon", "total"]):
        dif = tot - (der + hon)
        motivo = f"{cod}: Der. {der} + Hon. {hon} = {der + hon}, Total {tot} ({dif:+}) en {len(g)} filas"
        out.append(hallazgo("REDONDEO", REGLAS_VERSION, g, motivo, max(dif, Decimal(0)) * len(g)))
    return out


def guardar(hallazgos):
    """INSERT OR IGNORE: un hallazgo que ya existe no se duplica ni se pisa."""
    cols = ["hallazgo_id", "tipo", "regla_version", "fecha", "familia", "codigos", "veces",
            "monto_en_riesgo", "contiguas", "nro_ints", "ubicacion", "motivo"]
    with con:
        con.executemany(f"INSERT OR IGNORE INTO hallazgos ({', '.join(cols)}) "
                        f"VALUES ({', '.join('?' * len(cols))})",
                        [[sql_val(h[c]) for c in cols] for h in hallazgos])
        con.executemany("INSERT OR IGNORE INTO hallazgo_filas VALUES (?, ?)",
                        [(h["hallazgo_id"], f) for h in hallazgos for f in h["fila_ids"]])


ORDEN_TIPOS = ["NRO_INT_REPETIDO", "DUPLICADO_POTENCIAL", "REDONDEO"]
COLS_VISTA = ["tipo", "fecha", "familia", "codigos", "veces", "monto_en_riesgo", "contiguas",
              "nro_ints", "ubicacion", "motivo", "hallazgo_id"]


def vista(hallazgos):
    h = pd.DataFrame(hallazgos, columns=COLS_VISTA)
    h["_t"] = h["tipo"].map(ORDEN_TIPOS.index)
    h["_m"] = h["monto_en_riesgo"].map(float)
    return h.sort_values(["_t", "_m"], ascending=[True, False])[COLS_VISTA].reset_index(drop=True)


with medir("3 clasificación reglas") as m:
    HALLAZGOS = regla_nro_int(df) + regla_mismo_dia(df, freq) + regla_redondeo(df)
    guardar(HALLAZGOS)

control = []
for a, g in df.groupby("archivo"):
    suma, impreso = sum(importes(g["total"])), TOTALES.get(a)
    control.append({"archivo": a, "filas": len(g), "suma_filas": suma, "total_impreso": impreso,
                    "diferencia": impreso - suma if isinstance(impreso, Decimal) else None})
control = pd.DataFrame(control).set_index("archivo")
print("Control de totales por archivo:")
display(control)

H = vista(HALLAZGOS)
print(f"\n{len(H)} hallazgos potenciales:")
display(H.groupby("tipo", sort=False).agg(hallazgos=("hallazgo_id", "size"), filas=("veces", "sum"),
                                          monto_en_riesgo=("monto_en_riesgo", "sum")))
H

## 3b. Clasificación alternativa: LLM + skill (apagada por defecto)
El código arma los grupos candidatos (día + familia, 2+ filas, filtro laxo P(≥2) < 30 %), y el LLM decide cada grupo con la skill. Sirve cuando el criterio necesita más flexibilidad:
- semántica de descripciones truncadas,
- prácticas nuevas que las reglas no conocen,
- cambiar la política editando la skill, sin deploy.

Garantías:
- **Idempotencia:** respuesta cacheada por `hash(modelo, instrucciones, skill, candidatos)` y `temperature=0`.
- **Sin ids inventados:** se controla que todo id que devuelve el LLM esté entre los que recibió.
- **Trazabilidad:** los hallazgos los arma el código, con el mismo esquema que las reglas.

In [ ]:
# --- 3b-1. Skill ---------------------------------------------------------------------------------
SKILL_VERSION = "skill-duplicados-v1"
SKILL = """# Skill: duplicados potenciales (mismo criterio que el flujo de reglas)

Cada grupo es la misma práctica facturada 2 o más veces el mismo día. No hay datos del paciente.
Tu trabajo es aplicar esta regla, no tu criterio clínico.

## Regla
es_hallazgo = true solo si se cumplen las DOS condiciones:
1. La práctica NO es repetible por definición. Es repetible si:
   - algún código termina en X1 (medicamentos e insumos: se consolidan con la cantidad), o
   - la descripción dice CADA, SUBSIGUIENTE, SUBISGUIENTE, SEGUNDA/SEGUND o POR REGION
     (puede venir truncada, por ejemplo "POR REGIO").
2. Es rara: p_2_pct es menor que 5.
Si falla cualquiera de las dos, es_hallazgo = false.

## No cambian la decisión
- Que las filas estén en páginas o archivos distintos.
- Que haya 3 o más filas.
- Que los códigos sean distintos dentro del grupo (el sistema ya agrupó códigos equivalentes).
- El tipo de estudio, el servicio o tu intuición sobre si es el mismo paciente.

## Motivo
Una oración: la condición que se cumple o la que falla, con el valor de p_2_pct."""

INSTRUCCIONES = """Sos un auditor de facturación médica que aplica una regla fija descripta en la skill.
Devolvé una decisión por cada grupo_id recibido, sin omitir ni inventar grupos. Si es_hallazgo = true,
citá en `citas` todas las filas del grupo (archivo, pagina, nro_int) copiadas tal cual del input."""

In [ ]:
# --- 3b-2. Clasificación con LLM (con citas y guardrail) ------------------------------------------
from pydantic import BaseModel


class Cita(BaseModel):
    archivo: str
    pagina: int
    nro_int: str


class Decision(BaseModel):
    grupo_id: str
    es_hallazgo: bool
    motivo: str
    citas: list[Cita]


class Respuesta(BaseModel):
    decisiones: list[Decision]


SISTEMA = INSTRUCCIONES + """
Si es_hallazgo = true, citá en `citas` todas las filas del grupo (archivo, pagina, nro_int),
copiadas tal cual del input.

""" + SKILL


# La idea es acotar el input que recibe el LLM, el criterio para el challenge se mantiene simple,
# pero se puede ajustar según un criterio mínimo más estricto (ej: puede servir el flujo 3 como criterio mínimo)
def candidatos_llm(df, freq):
    """Grupos día + familia con 2+ filas y P(≥2) < UMBRAL_P_LLM. La skill decide el resto."""
    laxas = freq.index[freq["p_2"] < UMBRAL_P_LLM]
    out = []
    for (dia, fam), g in df[df["familia"].isin(laxas) & (df["dia"] != "")].groupby(["dia", "familia"]):
        if len(g) < 2:
            continue
        f = freq.loc[fam]
        out.append({"grupo_id": sha(dia, fam, n=8), "fecha": dia, "practica": f["practica"],
                    "codigos": sorted(set(g["codigo"])), "descripciones": sorted(set(g["descripcion"])),
                    "servicios": sorted(set(g["servicio"])), "veces": len(g),
                    "totales": [str(t) for t in g["total"]], "contiguas": contiguas(g),
                    "lambda": float(f["lambda"]), "p_2_pct": round(100 * float(f["p_2"]), 2),
                    "filas": [{"archivo": a, "pagina": int(p), "nro_int": n}
                              for a, p, n in zip(g["archivo"], g["pagina"], g["nro_int"])],
                    "_g": g})
    return out


PAGINAS_PDF = {x["archivo"]: len(x["rutas"]) for x in LOTE.values()}
FILAS_LOTE = set(zip(df["archivo"], df["pagina"].astype(int), df["nro_int"]))


def guardrail(d, por_id):
    """Motivos para rechazar una decisión (vacío = pasa): lo citado tiene que existir y ser del grupo."""
    c = por_id.get(d.grupo_id)
    if c is None:
        return ["grupo inexistente"]
    propias = {(f["archivo"], f["pagina"], f["nro_int"]) for f in c["filas"]}
    citadas = [(ct.archivo, ct.pagina, ct.nro_int) for ct in d.citas]
    motivos = []
    if d.es_hallazgo and set(citadas) != propias:
        motivos.append("no cita todas las filas del grupo")
    if len(citadas) != len(set(citadas)):
        motivos.append("cita filas repetidas")
    for archivo, pagina, nro in citadas:
        if archivo not in PAGINAS_PDF:
            motivos.append("archivo inexistente")
        elif not 1 <= pagina <= PAGINAS_PDF[archivo]:
            motivos.append("página inexistente")
        elif (archivo, pagina, nro) not in FILAS_LOTE:
            motivos.append("la fila no está en esa página")
        elif (archivo, pagina, nro) not in propias:
            motivos.append("fila de otro grupo")
    return sorted(set(motivos))


def llamar_llm(payload, m):
    """Gemini con salida estructurada; la respuesta queda en caché."""
    f = CACHE / f"llm_{sha(MODEL, SISTEMA, payload)}.json"
    if f.exists():
        m["cache_hits"] += 1
        return json.loads(f.read_text(encoding="utf-8"))
    from google import genai
    from google.genai import types
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION,
                          credentials=globals().get("credentials"))
    resp = client.models.generate_content(
        model=MODEL, contents=payload,
        config=types.GenerateContentConfig(system_instruction=SISTEMA,
                                           response_mime_type="application/json",
                                           response_schema=Respuesta, temperature=0))
    u = resp.usage_metadata
    r = {"text": resp.text, "tokens_in": u.prompt_token_count or 0,
         "tokens_out": (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)}
    f.write_text(json.dumps(r), encoding="utf-8")
    m["tokens_in"], m["tokens_out"] = r["tokens_in"], r["tokens_out"]
    m["costo_usd"] = (r["tokens_in"] * PRICE_IN + r["tokens_out"] * PRICE_OUT) / 1e6
    return r


HALLAZGOS_LLM = []
if not USAR_LLM:
    print("3b apagado (USAR_LLM = False): se usan las reglas.")
else:
    version_llm = f"llm {MODEL} {SKILL_VERSION}"
    reglas = {(h["fecha"], h["familia"]): h for h in HALLAZGOS if h["tipo"] == "DUPLICADO_POTENCIAL"}
    with medir("3b clasificación LLM") as m:
        cands = candidatos_llm(df, freq)
        payload = json.dumps([{k: v for k, v in c.items() if k != "_g"} for c in cands], ensure_ascii=False)
        r = llamar_llm(payload, m)
        decisiones = Respuesta.model_validate_json(r["text"]).decisiones
        por_id = {c["grupo_id"]: c for c in cands}
        devueltos = {d.grupo_id for d in decisiones}
        rechazos = {}
        for d in decisiones:
            if motivos := guardrail(d, por_id):
                rechazos[d.grupo_id] = (d, motivos)
                c = por_id.get(d.grupo_id)
                if c and (c["fecha"], c["practica"]) in reglas:   # respaldo: decide la regla
                    HALLAZGOS_LLM.append(reglas[(c["fecha"], c["practica"])])
                continue
            if d.es_hallazgo:
                c = por_id[d.grupo_id]
                HALLAZGOS_LLM.append(hallazgo("DUPLICADO_POTENCIAL", version_llm, c["_g"], f"[LLM] {d.motivo}",
                                              monto_en_riesgo(c["_g"]["total"]), c["fecha"], c["contiguas"]))
        guardar(HALLAZGOS_LLM)

    print(f"{len(cands)} candidatos · {len(HALLAZGOS_LLM)} hallazgos · sin decisión: {len(set(por_id) - devueltos)} · "
          f"rechazadas por el guardrail: {len(rechazos)} (esos grupos los decide la regla)")
    for gid, (d, motivos) in rechazos.items():
        print(f"   {gid}: {', '.join(motivos)}")
    print(f"Tokens: {r['tokens_in']:,} in / {r['tokens_out']:,} out"
          + (" (de caché: esta corrida no pagó)" if m["cache_hits"] else f" · USD {m['costo_usd']:.5f}"))

    # Reglas vs. LLM sobre los mismos grupos.
    llm = {d.grupo_id: d for d in decisiones}
    comp = pd.DataFrame([{
        "fecha": c["fecha"], "practica": c["practica"], "veces": c["veces"], "contiguas": c["contiguas"],
        "p_2_pct": c["p_2_pct"], "reglas": (c["fecha"], c["practica"]) in reglas,
        "llm": llm[c["grupo_id"]].es_hallazgo if c["grupo_id"] in llm else None,
        "guardrail": ", ".join(rechazos[c["grupo_id"]][1]) if c["grupo_id"] in rechazos else "ok",
        "motivo_llm": llm[c["grupo_id"]].motivo if c["grupo_id"] in llm else "",
    } for c in cands])
    display(pd.crosstab(comp["reglas"], comp["llm"], rownames=["reglas"], colnames=["llm"]))
    print("Desacuerdos:")
    display(comp[comp["reglas"] != comp["llm"]])
    display(vista(HALLAZGOS_LLM))

## 4. Cruzador (ficticio)
En producción, un LLM cruzaría cada hallazgo con la documentación respaldatoria (órdenes, informes) para descartar falsos positivos. En este challenge no tenemos esa documentación, así que pasan todos los hallazgos. Este bloque no registra métricas.

In [ ]:
# --- 4. Cruzador (ficticio) ----------------------------------------------------------------------
def cruzar(hallazgos, documentacion=None):
    return hallazgos   # sin documentación respaldatoria: no filtra


ACTIVOS = (HALLAZGOS if not USAR_LLM else
           [h for h in HALLAZGOS if h["tipo"] != "DUPLICADO_POTENCIAL"] + HALLAZGOS_LLM)
FINALES = cruzar(ACTIVOS)
print(f"{len(FINALES)} hallazgos pasan al trazador ({'LLM' if USAR_LLM else 'reglas'} para duplicados)")

## 5. Trazador
Marca sobre las páginas originales cada fila con hallazgos, usando las cajas que dio la extracción. Hay una caja por fila, y el color depende de lo que tiene esa fila:
- **rojo:** duplicado potencial (o Nro. Int. repetido)
- **celeste:** error de redondeo en la suma horizontal (Der. + Hon. ≠ Total)
- **naranja:** las dos cosas

Las marcas son **anotaciones** del PDF, así que la imagen original no se modifica. Al pasar el mouse por un recuadro se ven los motivos. El PDF de salida trae solo las páginas que tienen hallazgos.

In [ ]:
# --- 5. Trazador ---------------------------------------------------------------------------------
# Una caja por fila: rojo = duplicado potencial · celeste = redondeo (Der. + Hon. ≠ Total) · naranja = ambos
ROJO, CELESTE, NARANJA = (1, 0, 0), (0, 0.7, 1), (1, 0.5, 0)
TRAZAR = {"NRO_INT_REPETIDO", "DUPLICADO_POTENCIAL", "REDONDEO"}
DUPLICADOS = {"NRO_INT_REPETIDO", "DUPLICADO_POTENCIAL"}
LEYENDA = "rojo: duplicado potencial · celeste: redondeo · naranja: ambos"
SALIDA_PDF = OUT / "hallazgos_trazados.pdf"
PREVIEW = 2   # páginas a mostrar acá abajo


def color(tipos):
    if "REDONDEO" in tipos:
        return NARANJA if tipos & DUPLICADOS else CELESTE
    return ROJO


filas_por_id = df.set_index("fila_id")
marcas = defaultdict(lambda: defaultdict(list))   # (archivo, página) -> fila_id -> [hallazgos]
for h in FINALES:
    if h["tipo"] in TRAZAR:
        for fid in h["fila_ids"]:
            r = filas_por_id.loc[fid]
            marcas[(r["archivo"], int(r["pagina"]))][fid].append(h)

rutas = {x["archivo"]: x["path"] for x in LOTE.values()}
indice, conteo = [], {"rojo": 0, "celeste": 0, "naranja": 0}
with medir("5 trazador") as m, pymupdf.open() as out:
    for archivo in sorted({a for a, _ in marcas}):
        with pymupdf.open(rutas[archivo]) as src:
            for pagina in sorted(p for a, p in marcas if a == archivo):
                out.insert_pdf(src, from_page=pagina - 1, to_page=pagina - 1)
                page = out[-1]
                ancho, alto = page.rect.width, page.rect.height
                page.add_freetext_annot(pymupdf.Rect(10, 4, ancho / 2, 18), f"{archivo} · pág. {pagina}",
                                        fontsize=9, text_color=ROJO)
                page.add_freetext_annot(pymupdf.Rect(ancho / 2, 4, ancho - 10, 18), LEYENDA,
                                        fontsize=7, text_color=(0.3, 0.3, 0.3), align=2)
                for fid, hs in marcas[(archivo, pagina)].items():
                    r, c = filas_por_id.loc[fid], color({h["tipo"] for h in hs})
                    conteo[{ROJO: "rojo", CELESTE: "celeste", NARANJA: "naranja"}[c]] += 1
                    rect = pymupdf.Rect(r["x0"] * ancho - 3, r["y0"] * alto - 2,
                                        r["x1"] * ancho + 3, r["y1"] * alto + 2) * page.derotation_matrix
                    a = page.add_rect_annot(rect)
                    a.set_colors(stroke=c)
                    a.set_border(width=1.5)
                    a.set_info(title=", ".join(h["hallazgo_id"] for h in hs),
                               content="\n".join(h["motivo"] for h in hs))
                    a.update()
                    page.add_freetext_annot(pymupdf.Rect(rect.x0, rect.y0 - 10, rect.x0 + 320, rect.y0),
                                            " · ".join(f"{h['tipo']} {h['hallazgo_id'][:8]}" for h in hs),
                                            fontsize=6, text_color=c)
                indice.append({"pagina_salida": len(out), "archivo": archivo, "pagina": pagina,
                               "hallazgos": ", ".join(sorted({h["hallazgo_id"][:8]
                                                              for hs in marcas[(archivo, pagina)].values() for h in hs}))})
    m["paginas"] = len(out)
    previews = [out[i].get_pixmap(dpi=70).tobytes("png") for i in range(min(PREVIEW, len(out)))]
    if len(out):
        out.save(SALIDA_PDF)

if not indice:
    print("Sin hallazgos para trazar.")
else:
    print(f"{SALIDA_PDF}: {len(indice)} páginas · filas marcadas: {conteo}")
    display(pd.DataFrame(indice).set_index("pagina_salida"))
    for png in previews:
        display(Image(png))

## Métricas de producción
Latencia, páginas, tokens y costo por bloque. El costo es lo que efectivamente se facturó en esta corrida: lo que vino de caché cuenta USD 0.

In [ ]:
# --- Métricas ------------------------------------------------------------------------------------
met = pd.DataFrame(METRICAS.values())
total = met[["latencia_s", "tokens_in", "tokens_out", "costo_usd", "cache_hits"]].sum()
total["paginas"] = sum(len(x["rutas"]) for x in LOTE.values())   # páginas del lote, no la suma de bloques
met = pd.concat([met, pd.DataFrame([{"bloque": "TOTAL", "detalle": "", **total}])], ignore_index=True)
met = met.astype({c: int for c in ["paginas", "tokens_in", "tokens_out", "cache_hits"]})
met["costo_usd"] = met["costo_usd"].round(5)
display(met.set_index(["bloque", "detalle"]))

print(f"Por página: {total['latencia_s'] / total['paginas']:.2f}s · "
      f"USD {total['costo_usd'] / total['paginas']:.5f} en esta corrida "
      f"(sin caché, el OCR solo es USD {PRICE_OCR:.4f}/página → ~USD {PRICE_OCR * 1e6:,.0f} por 1M páginas)")
met.to_csv(OUT / "metricas.csv", index=False)

## Salidas

In [ ]:
# --- Salidas -------------------------------------------------------------------------------------
vista(FINALES).to_csv(OUT / "hallazgos.csv", index=False)
n_db = con.execute("SELECT COUNT(*) FROM hallazgos").fetchone()[0]
print(f"hallazgos.csv          {OUT / 'hallazgos.csv'}  ({len(FINALES)} hallazgos de esta corrida)")
print(f"hallazgos_trazados.pdf {SALIDA_PDF}")
print(f"metricas.csv           {OUT / 'metricas.csv'}")
print(f"base SQLite            {DB}  ({n_db} hallazgos guardados; re-ejecutar no los duplica)")